# IAA Hackathon 2026 — Notebook 1: Data Prep & Claims Development

**Purpose:** Load parquet files, clean data, develop claims to ultimate using external benchmark development factors, join enrichment tables, and save processed data for modelling.

**AI-assisted:** Genie Code generated code structure, development factor benchmarks, and documentation.
**Human decisions:** Feature selection (Features_v1.xlsx), development factor assumptions reviewed by pricing actuary, exposure formula, construction_type mapping.

Data source: https://www.kaggle.com/competitions/iaa-ai-assisted-hackathon-2026-part-1/overview

In [0]:
import pandas as pd
import numpy as np
import os

# Paths (relative to notebooks_v3 folder)
DATA_DIR = '../data'
PROC_DIR = '../data/proc'
EXCEL_DIR = '../Excel files_v3'
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(EXCEL_DIR, exist_ok=True)

# Load all parquet files as pandas DataFrames
files = [
    'policies_motor', 'policies_buildings', 'policies_contents',
    'test_motor', 'test_buildings', 'test_contents',
    'claims', 'customers', 'addresses', 'weather',
    'sample_submission'
]

data = {}
for f in files:
    data[f] = pd.read_parquet(f'{DATA_DIR}/{f}.parquet')
    print(f'{f}: {data[f].shape[0]:,} rows x {data[f].shape[1]} cols')

In [0]:
# --- Data Cleaning ---

policy_test_files = [
    'policies_motor', 'policies_buildings', 'policies_contents',
    'test_motor', 'test_buildings', 'test_contents'
]

# Fix exposure = (period_end - period_start + 1) / 365
# Create Year field from period_start
for name in policy_test_files:
    df = data[name]
    df['period_start'] = pd.to_datetime(df['period_start'])
    df['period_end'] = pd.to_datetime(df['period_end'])
    df['exposure'] = (df['period_end'] - df['period_start']).dt.days + 1
    df['exposure'] = df['exposure'] / 365.0
    df['Year'] = df['period_start'].dt.year

# Fix construction_type in buildings (normalise case, map timber codes)
# Timber was 'T' to 2022, 'W' from 2023 — map both to TIMBER
for name in ['policies_buildings', 'test_buildings']:
    df = data[name]
    if 'construction_type' in df.columns:
        df['construction_type'] = df['construction_type'].astype(str).str.upper().str.strip()
        df['construction_type'] = df['construction_type'].replace({
            'T': 'TIMBER', 'W': 'TIMBER', 'NON-STANDARD': 'NON_STANDARD'
        })

# Fix motor customer_id left-padding to 8 chars
# Without this, ~10% of motor rows don't join to customers
for name in ['policies_motor', 'test_motor']:
    df = data[name]
    df['customer_id'] = df['customer_id'].astype(str).str.zfill(8)

print('Data cleaning complete.')
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    df = data[name]
    print(f'  {name}: exposure {df["exposure"].min():.4f}–{df["exposure"].max():.4f}, '
          f'Year {df["Year"].min()}–{df["Year"].max()}')
print(f'  Buildings construction_type: {data["policies_buildings"]["construction_type"].value_counts().to_dict() if "construction_type" in data["policies_buildings"].columns else "N/A"}')

In [0]:
# --- Claims Processing: Develop Incurred to Ultimate ---

claims = data['claims'].copy()

# Exclude repudiated (declined) claims
claims = claims[claims['status'] != 'repudiated'].copy()
print(f'Claims after excluding repudiated: {len(claims):,}')

# Create incurred_net_of_recoveries
claims['incurred_net_of_recoveries'] = claims['incurred_to_date'] - claims['recoveries']

# Extract accident year from loss_date
claims['loss_date'] = pd.to_datetime(claims['loss_date'])
claims['accident_year'] = claims['loss_date'].dt.year

# Development age at valuation date (31 Dec 2024)
claims['dev_age'] = 2024 - claims['accident_year']

# --- External Benchmark Development Factors by Peril ---
# Cumulative % developed at each age (years from accident to valuation 31 Dec 2024)
#
# Source: General insurance market development patterns for developed markets
# (UK/industry benchmarks, adapted for this synthetic portfolio)
#
# KEY ASSUMPTION: With only 4 years of data (2021-2024), long-tail perils like
# bodily injury (10+ years to develop) and subsidence (5+ years) are NOT fully
# developed even for 2021 claims. External benchmarks are essential — the data
# alone cannot capture the full development tail.
#
# Rationale:
# - bodily_injury: Very long tail. 30% at age 0, 50% at 1, 65% at 2, 75% at 3.
#   Can take 10+ years to fully develop (IBNR + IBNER + reopenings).
# - subsidence: Long tail, seasonal. 35% at 0, 55% at 1, 70% at 2, 82% at 3.
# - own_damage/theft/glass_hail: Fast developing, 85-95%+ at age 0.
# - storm/flood: Medium, affected by event clustering and reconstruction delays.

dev_pct = {
    # Motor perils
    'own_damage':       {0: 0.90, 1: 0.97, 2: 0.99, 3: 1.00},
    'third_party_prop': {0: 0.80, 1: 0.93, 2: 0.98, 3: 1.00},
    'bodily_injury':    {0: 0.30, 1: 0.50, 2: 0.65, 3: 0.75},  # Very long tail
    'theft':            {0: 0.85, 1: 0.96, 2: 0.99, 3: 1.00},
    'glass_hail':       {0: 0.95, 1: 0.99, 2: 1.00, 3: 1.00},
    # Buildings perils
    'storm':            {0: 0.75, 1: 0.90, 2: 0.96, 3: 1.00},
    'flood':            {0: 0.60, 1: 0.80, 2: 0.90, 3: 0.96},
    'escape_of_water':  {0: 0.75, 1: 0.90, 2: 0.96, 3: 1.00},
    'subsidence':       {0: 0.35, 1: 0.55, 2: 0.70, 3: 0.82},  # Long tail
    'fire':             {0: 0.80, 1: 0.93, 2: 0.97, 3: 1.00},
    'accidental_damage':{0: 0.85, 1: 0.95, 2: 0.99, 3: 1.00},
    # Contents perils reuse same keys: theft, escape_of_water, accidental_damage, flood
}

# Apply: ultimate = incurred_net_of_recoveries / %_developed
def get_dev_factor(row):
    peril = row['peril']
    age = int(row['dev_age'])
    if peril in dev_pct:
        pct = dev_pct[peril].get(age, dev_pct[peril].get(3, 1.0))
        return 1.0 / pct if pct > 0 else 1.0
    return 1.0  # Unknown peril: no development

claims['dev_factor'] = claims.apply(get_dev_factor, axis=1)
claims['ultimate_claims'] = claims['incurred_net_of_recoveries'] * claims['dev_factor']

# Summary
inc_total = claims['incurred_net_of_recoveries'].sum()
ult_total = claims['ultimate_claims'].sum()
print(f'\nClaims development summary:')
print(f'  incurred_net_of_recoveries: {inc_total:,.0f}')
print(f'  ultimate_claims: {ult_total:,.0f}')
print(f'  Overall uplift: {(ult_total/inc_total - 1)*100:.1f}%')
print(f'\nDevelopment factors by peril (age 0=AY2024, age 3=AY2021):')
for peril in sorted(dev_pct.keys()):
    factors = {a: round(1.0/p, 2) for a, p in sorted(dev_pct[peril].items())}
    print(f'  {peril:20s}: {factors}')

In [0]:
# --- Match Claims to Policy Rows and Aggregate to Row Level ---
# Match each claim to the policy row where loss_date falls within [period_start, period_end]
# This correctly handles the "multiple rows per policy per year" case.

processed = {}

for line_name, pol_name in [('motor', 'policies_motor'),
                             ('buildings', 'policies_buildings'),
                             ('contents', 'policies_contents')]:
    pol = data[pol_name].copy()
    line_claims = claims[claims['line'] == line_name].copy()

    # Merge claims to policies on policy_id
    merged = line_claims.merge(
        pol[['row_id', 'policy_id', 'period_start', 'period_end', 'exposure']],
        on='policy_id', how='inner'
    )

    # Filter to where loss_date is within the policy period
    mask = (merged['loss_date'] >= merged['period_start']) & (merged['loss_date'] <= merged['period_end'])
    matched = merged[mask].copy()
    unmatched = len(line_claims) - matched['claim_id'].nunique()

    # Handle claims matching multiple rows (allocate by exposure)
    claim_match_count = matched.groupby('claim_id')['row_id'].nunique()
    multi = claim_match_count[claim_match_count > 1]
    if len(multi) > 0:
        matched['weight'] = matched.groupby('claim_id')['exposure'].transform(lambda x: x / x.sum())
        matched['ultimate_alloc'] = matched['ultimate_claims'] * matched['weight']
        print(f'{line_name}: {len(multi)} claims matched multiple rows (allocated by exposure)')
    else:
        matched['ultimate_alloc'] = matched['ultimate_claims']

    # Aggregate per row_id
    claim_agg = matched.groupby('row_id').agg(
        ultimate_claims=('ultimate_alloc', 'sum'),
        claim_count=('claim_id', 'nunique')
    ).reset_index()

    # Merge back to policies (left join — rows with no claims get 0)
    pol = pol.merge(claim_agg, on='row_id', how='left')
    pol['ultimate_claims'] = pol['ultimate_claims'].fillna(0)
    pol['claim_count'] = pol['claim_count'].fillna(0).astype(int)

    processed[line_name] = pol
    n_claims = (pol['claim_count'] > 0).sum()
    print(f'{line_name}: {len(pol):,} rows | {n_claims:,} with claims ({n_claims/len(pol)*100:.1f}%) | '
          f'ultimate: {pol["ultimate_claims"].sum():,.0f} | unmatched: {unmatched}')

print('\nClaims matching complete.')

In [0]:
# --- Join Customers, Addresses, Weather to All Policy and Test Files ---

customers = data['customers'].copy()
addresses = data['addresses'].copy()
weather = data['weather'].copy()

# Check available years in weather data
weather_years = set(weather['year'].unique())
latest_weather_year = max(weather_years)
print(f'Weather years: {sorted(weather_years)}')
if 2025 not in weather_years:
    print(f'  2025 not available — using {latest_weather_year} for test data')

def enrich(df):
    """Join customers, addresses, and weather to a policy/test dataframe."""
    # Join customers on customer_id
    df = df.merge(customers, on='customer_id', how='left', suffixes=('', '_cust'))
    # Join addresses on address_id
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    # Join weather on address_id + year (use latest available for test data)
    df['_wy'] = df['Year'].map(lambda y: y if y in weather_years else latest_weather_year)
    df = df.merge(weather, left_on=['address_id', '_wy'], right_on=['address_id', 'year'], how='left', suffixes=('', '_w'))
    df = df.drop(columns=['_wy'])
    if 'year' in df.columns:
        df = df.drop(columns=['year'])
    return df

# Enrich training data
for line_name in ['motor', 'buildings', 'contents']:
    processed[line_name] = enrich(processed[line_name])
    print(f'train_{line_name}: {processed[line_name].shape[0]:,} x {processed[line_name].shape[1]} cols')

# Enrich test data
test_processed = {}
for line_name, test_name in [('motor', 'test_motor'), ('buildings', 'test_buildings'), ('contents', 'test_contents')]:
    test_processed[line_name] = enrich(data[test_name].copy())
    print(f'test_{line_name}: {test_processed[line_name].shape[0]:,} x {test_processed[line_name].shape[1]} cols')

In [0]:
%pip install -q openpyxl

In [0]:
# --- Trend Analysis: Ultimate Claims Rate by Year ---

print('=== Ultimate Claims Rate by Year and Line ===\n')
trend_rows = []
for line_name in ['motor', 'buildings', 'contents']:
    df = processed[line_name]
    for yr in sorted(df['Year'].unique()):
        ydf = df[df['Year'] == yr]
        total_ult = ydf['ultimate_claims'].sum()
        total_exp = ydf['exposure'].sum()
        rate = total_ult / total_exp if total_exp > 0 else 0
        freq = (ydf['claim_count'] > 0).mean() * 100
        trend_rows.append({
            'line': line_name, 'year': yr, 'rows': len(ydf),
            'exposure': round(total_exp), 'ultimate_claims': round(total_ult),
            'claims_rate': round(rate, 2), 'freq_pct': round(freq, 2)
        })
        print(f'  {line_name} {yr}: rate={rate:.2f} | ult={total_ult:,.0f} | exp={total_exp:,.0f} | freq={freq:.1f}%')

trend_df = pd.DataFrame(trend_rows)

# Assess trend (development factors already correct for underdevelopment)
print('\n=== Trend Assessment ===')
for line_name in ['motor', 'buildings', 'contents']:
    lt = trend_df[trend_df['line'] == line_name].sort_values('year')
    rates = lt['claims_rate'].values
    if len(rates) >= 2:
        change = (rates[-1] / rates[0] - 1) * 100
        direction = 'upward' if rates[-1] > rates[0] else 'downward' if rates[-1] < rates[0] else 'flat'
        print(f'  {line_name}: {direction} ({change:+.1f}% over {lt["year"].min()}-{lt["year"].max()})')

# --- Excel Checkpoint ---
# Development factors table
dev_rows = []
for peril, ages in dev_pct.items():
    row = {'peril': peril}
    for age in sorted(ages):
        row[f'age_{age}_pct'] = ages[age]
        row[f'age_{age}_factor'] = round(1.0/ages[age], 2)
    dev_rows.append(row)
dev_df = pd.DataFrame(dev_rows)

# Claims by peril
peril_rows = []
for line_name in ['motor', 'buildings', 'contents']:
    lc = claims[claims['line'] == line_name]
    for peril in sorted(lc['peril'].unique()):
        pc = lc[lc['peril'] == peril]
        inc_sum = pc['incurred_net_of_recoveries'].sum()
        peril_rows.append({
            'line': line_name, 'peril': peril, 'claims': len(pc),
            'incurred_net': round(inc_sum),
            'ultimate': round(pc['ultimate_claims'].sum()),
            'mean_dev_factor': round(pc['dev_factor'].mean(), 2),
            'uplift_pct': round((pc['ultimate_claims'].sum()/inc_sum - 1)*100, 1) if inc_sum != 0 else 0
        })
peril_df = pd.DataFrame(peril_rows)

excel_path = f'{EXCEL_DIR}/claims_development_checkpoint.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    dev_df.to_excel(writer, sheet_name='Development Factors', index=False)
    peril_df.to_excel(writer, sheet_name='Claims by Peril', index=False)
    trend_df.to_excel(writer, sheet_name='Trend Analysis', index=False)
print(f'\nExcel checkpoint saved: {excel_path}')

# --- Save Processed Data for Modelling Notebooks ---
for line_name in ['motor', 'buildings', 'contents']:
    processed[line_name].to_parquet(f'{PROC_DIR}/train_{line_name}.parquet', index=False)
    test_processed[line_name].to_parquet(f'{PROC_DIR}/test_{line_name}.parquet', index=False)
    print(f'Saved: train_{line_name} ({processed[line_name].shape}) + test_{line_name} ({test_processed[line_name].shape})')

data['sample_submission'].to_parquet(f'{PROC_DIR}/sample_submission.parquet', index=False)
print('All processed data saved to data/proc/')

## Summary

### Key Findings
- Claims developed to ultimate using external benchmark development factors by peril
- Long-tail perils (bodily injury, subsidence) show significant uplift even for 2021 claims
- Trend analysis shows whether 2021-2024 ultimate claims rates need projecting to 2025
- Development factors already correct for underdevelopment — any remaining trend is genuine

### Assumptions (External Benchmark Development Factors)
- **bodily_injury**: only 75% developed at age 3 (10+ years to fully develop)
- **subsidence**: only 82% developed at age 3 (5+ years to fully develop)
- Short-tail perils (own_damage, theft, glass_hail): 90-95%+ developed at age 0
- Claims valued at 31 Dec 2024; development age = 2024 - accident_year

### AI / Human Log
- **AI (Genie Code):** Generated code, development factor benchmarks, data cleaning, matching logic
- **Human (Pricing Actuary):** Reviewed development assumptions, confirmed BI long tail, approved feature selection, specified exposure formula and construction_type mapping

### Files Output
- `data/proc/train_{motor,buildings,contents}.parquet` — processed training data
- `data/proc/test_{motor,buildings,contents}.parquet` — processed test data
- `Excel files_v3/claims_development_checkpoint.xlsx` — development factors, claims by peril, trend